# Inconsistent condition

The story is split into blocks (a line with `###` between them). Over the **whole story**:
- **consonants**: every t and every k is replaced by one of its alternatives, drawn at random (33 % each by default; one draw per word, so a word with two t's uses the same alternative for both)
- **vowels**: exactly the number of **words** and **phonemes** you choose for each vowel is changed, the words are drawn at random over the whole story, and a word never gets more than one vowel change

Each block can be spoken by a **different voice**, and the timing can be taken from a **reference voice reading the original text** (so every voice and every condition has the same timing).

The output is written to disk (an HTML player and a wav per block and for the whole condition); nothing is displayed in the notebook.

All functions are in `accent_tools.py` (next to `functions.py`); `help(make_inconsistent)`, `help(render_condition)` … show their documentation.

In [ ]:
from accent_tools import *

## Settings
Everything you need to change is in this cell.

In [ ]:
# ---- story --------------------------------------------------------------------------------------------
STORY_FILE = "story1.txt"     # blocks separated by a line starting with ### (e.g. "###" or "### block 2")
STORY_NAME = None             # name used in titles and file names (None = file name, e.g. "story1")
BLOCK_SEP  = "###"

# ---- voices -------------------------------------------------------------------------------------------
BLOCK_VOICES = ["im_nicola", "if_sara", "im_nicola"]   # one voice per block (or one string for all blocks)
REF_VOICE    = "im_nicola"                             # reference voice for the timing (see below), or None
# your own voice files: give each a name (shown in titles and summaries) as (name, tensor)
#   import torch; from pathlib import Path; VOICES_DIR = Path("custom_voices")
#   BLOCK_VOICES = [("nicola_20", torch.load(VOICES_DIR / "nicola_20.pt", weights_only=True)),
#                   ("nicola_50", torch.load(VOICES_DIR / "nicola_50.pt", weights_only=True)),
#                   ("nicola_80", torch.load(VOICES_DIR / "nicola_80.pt", weights_only=True))]

# ---- timing and prosody --------------------------------------------------------------------------
# The reference voice reads the ORIGINAL (unmanipulated) text; its timing is put on every block.
#   None             -> no transfer: every block voice times the accented text itself
#   "im_nicola"      -> the same reference voice for all blocks
#   [v1, v2, v3]     -> one reference voice per block
TRANSFER = dict(duration=True,   # take the reference voice's durations
                pitch=False,     # take its pitch contour too (moved into the block voice's range)
                energy=False)    # take its energy contour too
DUR_MODE = "raw"     # "raw":  exactly the reference voice's frames -> identical timing for every voice/condition
                     # "rate": the reference rhythm, stretched to each block voice's own speaking rate
# In a palatalized consonant (kʲ, tʲ ...) the consonant gets the reference duration and the added ʲ keeps
# the block voice's own duration (KEEP_OWN). Add "ʰ" to treat aspiration the same way, or () to split.
KEEP_OWN = ("ʲ",)

# ---- errors -------------------------------------------------------------------------------------------
# consonants: alternatives as a list (equal chance) or a dict {alternative: weight}
CONSONANTS = {"t": ["tʲ", "θ", "ɖ"],
              "k": ["x", "ɡ", "kʲ"]}               # ɡ = U+0261, the g in Kokoro's vocab (not the keyboard g)
SAME_IN_WORD = True                                # all t's (k's) of one word get the same alternative

# vowels: replacement, number of WORDS and number of PHONEMES to change over the whole story
VOWELS = {"e": dict(new="ø", words=290, phonemes=320),
          "i": dict(new="y", words=223, phonemes=246),
          "o": dict(new="ə", words=248, phonemes=304)}
OCCURRENCES = "all"      # "all":     every e of a chosen word changes; phonemes reached by choosing words with
                         #            more / fewer e's (exact unless impossible)
                         # "partial": exactly `phonemes` changes spread over the chosen words (some words keep
                         #            one of their e's)
SEED = 1                 # same seed -> same words and same alternatives

# ---- measurement (printed in the summary and saved in the _summary.csv) -----------------------------
# weight of each substitution for the weighted distance pw_w; pairs not listed = number of features
WEIGHTS = {("t", "tʲ"): 1, ("t", "θ"): 2, ("t", "ɖ"): 2,
           ("k", "x"):  1, ("k", "ɡ"): 1, ("k", "kʲ"): 1,
           ("e", "ø"):  1, ("i", "y"): 1, ("o", "ə"):  3}

# ---- output -------------------------------------------------------------------------------------------
OUT_DIR       = "output_online/inconsistent"
EXPORT_NATIVE = False                       # also export a native version with the same voices and timing
NATIVE_DIR    = "output_online/native"
N_PEEK        = 2                           # sentences per block shown in the IPA preview

## 1. Load the story
Splits the story into blocks at the `###` lines, converts every sentence to IPA and turns every ɪ into i.
The printout shows the number of sentences and words per block, check that it matches your blocks.

In [ ]:
story = load_story(STORY_FILE, name=STORY_NAME, block_sep=BLOCK_SEP)

## 2. Manipulate
Consonants first, then the vowel quotas. The printout shows, for every vowel, how many candidate words the story has and how many words / phonemes were changed (with a warning if a quota cannot be reached).
The summary table then shows, per block, what changed and how much (`pw` = per-word normalized distance, every change = 1; `pw_w` = with your `WEIGHTS`; both in %, see `help(measure_distance)`).

In [ ]:
cond = make_inconsistent(story, consonants=CONSONANTS, vowels=VOWELS, same_in_word=SAME_IN_WORD,
                         occurrences=OCCURRENCES, seed=SEED)
print_summary(cond, weights=WEIGHTS);

### Look at the accented IPA (optional)
The first sentences of every block, native above accented. All sentences are also saved in `_ipa.txt` at export.

In [ ]:
for blk in cond["blocks"]:
    print(f"── block {blk['index']}")
    for nat, man in list(zip(blk["ipa"], blk["manip"]))[:N_PEEK]:
        print("   native  :", nat)
        print("   accented:", man)

## 3. Synthesize
One call per block. This is the slow step (a few seconds per sentence on CPU).

In [ ]:
results = render_condition(cond, voices=BLOCK_VOICES, ref_voice=REF_VOICE, **TRANSFER,
                           dur_mode=DUR_MODE, keep_own=KEEP_OWN)

## 4. Export
Writes everything to `OUT_DIR`; nothing is displayed here. Open the `.html` files in a browser.

| file | content |
|---|---|
| `<condition>_<story>_block1.html` … | interactive player of one block (word boxes, token durations, altered phonemes highlighted, pitch / energy panel, legend of the changes in that block) |
| `<condition>_<story>_block1.wav` … | audio of one block |
| `<condition>_<story>_full.html` / `.wav` | the whole condition, blocks one after the other |
| `<condition>_<story>_ipa.txt` | native and accented IPA of every sentence |
| `<condition>_<story>_summary.csv` | per block: voice, reference voice, duration, words, changes, distances |

In [ ]:
export_condition(cond, results, OUT_DIR, weights=WEIGHTS);

## 5. (Optional) Native version with the same voices and timing
The same story without any error, with the same block voices and the same reference timing, so it can be compared with the accented condition one to one.

In [ ]:
if EXPORT_NATIVE:
    native = make_native(story)
    native_results = render_condition(native, voices=BLOCK_VOICES, ref_voice=REF_VOICE, **TRANSFER,
                                      dur_mode=DUR_MODE, keep_own=KEEP_OWN)
    export_condition(native, native_results, NATIVE_DIR);